In [10]:
import os
import shutil
import yaml
import random
from glob import glob
from sklearn.model_selection import train_test_split

# ================= 設定區 =================
INCLUDE_BACKGROUND = False           # ✨ 是否要加入背景圖片的開關 (True/False)

SOURCE_IMG_DIR = "./labelimg-master/trainimg"
SOURCE_LABEL_DIR = "./labelimg-master/trainimg"
BACKGROUND_DIR = "./backround_photo"
CLASSES_FILE = "./labelimg-master/trainimg/classes.txt"

BASE_PROJECT_DIR = "debug_chunks_configurable"
CHUNK_SIZE = 50
TRAIN_RATIO = 0.8
SEED = 42

random.seed(SEED)
# ========================================

# 1. 讀取類別名稱
class_names = []
if os.path.exists(CLASSES_FILE):
    with open(CLASSES_FILE, 'r', encoding='utf-8') as f:
        class_names = [line.strip() for line in f if line.strip()]
else:
    print(f"❌ 找不到類別檔 {CLASSES_FILE}")
    class_names = ["item"]

# 2. 蒐集圖片路徑
source_imgs = glob(os.path.join(SOURCE_IMG_DIR, "*.jpg"))
bg_imgs = []

if INCLUDE_BACKGROUND:
    bg_imgs = glob(os.path.join(BACKGROUND_DIR, "*.jpg"))
    print(f"📂 背景圖功能已【開啟】，偵測到 {len(bg_imgs)} 張背景圖。")
else:
    print(f"🚫 背景圖功能已【關閉】，僅處理標註圖片。")

# 混合並隨機排序
all_files = source_imgs + bg_imgs
random.shuffle(all_files)

if not all_files:
    print("❌ 找不到任何圖片，請檢查路徑設定！")
    exit()

print(f"📊 總計圖片: {len(all_files)} 張，預計切分為 {len(all_files)//CHUNK_SIZE + 1} 塊\n")

# 3. 開始分塊處理
for i in range(0, len(all_files), CHUNK_SIZE):
    chunk_id = (i // CHUNK_SIZE) + 1
    chunk_files = all_files[i : i + CHUNK_SIZE]
    
    chunk_dir = os.path.join(BASE_PROJECT_DIR, f"chunk_{chunk_id}")
    dirs = {
        'train_img': os.path.join(chunk_dir, "train", "images"),
        'train_lbl': os.path.join(chunk_dir, "train", "labels"),
        'val_img': os.path.join(chunk_dir, "val", "images"),
        'val_lbl': os.path.join(chunk_dir, "val", "labels")
    }
    for d in dirs.values(): os.makedirs(d, exist_ok=True)

    # 4. 區分 Train / Val
    train_list, val_list = train_test_split(chunk_files, test_size=(1 - TRAIN_RATIO), random_state=SEED)

    def process_copy(file_list, target_img_dir, target_lbl_dir):
        count_bg = 0
        for img_path in file_list:
            fname = os.path.basename(img_path)
            basename = os.path.splitext(fname)[0]
            shutil.copy(img_path, os.path.join(target_img_dir, fname))
            
            # 判斷邏輯：如果是背景圖或是開關開啟且路徑匹配
            is_bg_file = (INCLUDE_BACKGROUND and BACKGROUND_DIR in img_path.replace("\\", "/"))
            lbl_name = basename + ".txt"
            
            if is_bg_file:
                # 建立空標籤
                with open(os.path.join(target_lbl_dir, lbl_name), 'w') as f: pass
                count_bg += 1
            else:
                # 尋找原始標籤
                src_lbl = os.path.join(SOURCE_LABEL_DIR, lbl_name)
                if os.path.exists(src_lbl):
                    shutil.copy(src_lbl, os.path.join(target_lbl_dir, lbl_name))
                else:
                    # 找不到標籤則視為空背景，避免 YOLO 訓練中斷
                    with open(os.path.join(target_lbl_dir, lbl_name), 'w') as f: pass
                    count_bg += 1
        return count_bg

    bg_in_train = process_copy(train_list, dirs['train_img'], dirs['train_lbl'])
    bg_in_val = process_copy(val_list, dirs['val_img'], dirs['val_lbl'])

    # 5. 生成 data.yaml
    yaml_data = {'train': 'train/images', 'val': 'val/images', 'nc': len(class_names), 'names': class_names}
    with open(os.path.join(chunk_dir, "data.yaml"), 'w', encoding='utf-8') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, allow_unicode=True)

    print(f"✅ Chunk {chunk_id} 完成 | 總計: {len(chunk_files)} 張 (含空標籤: {bg_in_train + bg_in_val} 張)")

print(f"\n✨ 任務結束，資料夾：{BASE_PROJECT_DIR}")

🚫 背景圖功能已【關閉】，僅處理標註圖片。
📊 總計圖片: 400 張，預計切分為 9 塊

✅ Chunk 1 完成 | 總計: 50 張 (含空標籤: 0 張)
✅ Chunk 2 完成 | 總計: 50 張 (含空標籤: 0 張)
✅ Chunk 3 完成 | 總計: 50 張 (含空標籤: 0 張)
✅ Chunk 4 完成 | 總計: 50 張 (含空標籤: 0 張)
✅ Chunk 5 完成 | 總計: 50 張 (含空標籤: 0 張)
✅ Chunk 6 完成 | 總計: 50 張 (含空標籤: 0 張)
✅ Chunk 7 完成 | 總計: 50 張 (含空標籤: 0 張)
✅ Chunk 8 完成 | 總計: 50 張 (含空標籤: 0 張)

✨ 任務結束，資料夾：debug_chunks_configurable


In [11]:
import os
import pandas as pd
from ultralytics import YOLO
from glob import glob

# ================= 設定區 =================
CHUNKS_PATH = "debug_chunks_configurable"  
MODEL_VARIANT = "yolov10s.pt"             
EPOCHS = 30                                
BATCH_SIZE = 32
DEVICE = 0                                 # RTX 4060 Ti
# ========================================

def run_chunk_diagnostic():
    chunk_dirs = sorted(glob(os.path.join(CHUNKS_PATH, "chunk_*")))
    results_list = []

    print(f"🚀 啟動雙指標診斷程序，目標：{len(chunk_dirs)} 個分塊...")

    for chunk_dir in chunk_dirs:
        chunk_name = os.path.basename(chunk_dir)
        yaml_path = os.path.join(chunk_dir, "data.yaml")
        
        if not os.path.exists(yaml_path):
            continue

        print(f"🔎 正在測試 {chunk_name}...")
        model = YOLO(MODEL_VARIANT)
        
        results = model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            device=DEVICE,
            project=os.path.join(CHUNKS_PATH, "test_results"),
            name=chunk_name,
            verbose=False,
            plots=False
        )

        # 提取核心指標
        map50 = results.results_dict.get('metrics/mAP50(B)', 0)
        map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 0)
        
        results_list.append({
            "Chunk": chunk_name,
            "mAP50": round(map50, 4),
            "mAP50-95": round(map50_95, 4),
            "Gap": round(map50 - map50_95, 4) # 差距越大，代表標註框越不精準
        })

    # 排序邏輯：優先看 mAP50-95
    df = pd.DataFrame(results_list)
    df = df.sort_values(by="mAP50-95", ascending=True)

    print("\n" + "="*50)
    print("🏆 Chunk 標註品質診斷表 (依 mAP50-95 排序)")
    print("="*50)
    print(df.to_string(index=False))
    print("="*50)
    
    df.to_csv(os.path.join(CHUNKS_PATH, "detailed_quality_report.csv"), index=False)
    print(f"📊 完整報告已產出。")

if __name__ == "__main__":
    run_chunk_diagnostic()

🚀 啟動雙指標診斷程序，目標：8 個分塊...
🔎 正在測試 chunk_1...
New https://pypi.org/project/ultralytics/8.4.7 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.230  Python-3.10.19 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=debug_chunks_configurable\chunk_1\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mos

In [8]:
import os
import shutil
from glob import glob

# ================= 設定區 (請確認與之前腳本一致) =================
TARGET_CHUNK = "chunk_4"
CHUNKS_BASE_PATH = "debug_chunks_configurable"

# 原始源頭路徑
SOURCE_IMG_DIR = "./labelimg-master/trainimg"
BACKGROUND_DIR = "./backround_photo"

# 建立一個暫存資料夾，把要刪除的檔案先放這裡 (比較安全)
TRASH_DIR = "./dataset_trash_bin"
os.makedirs(TRASH_DIR, exist_ok=True)
# =============================================================

def cleanup_source_files():
    chunk_path = os.path.join(CHUNKS_BASE_PATH, TARGET_CHUNK)
    
    if not os.path.exists(chunk_path):
        print(f"❌ 找不到分塊資料夾: {chunk_path}")
        return

    # 1. 蒐集 chunk_12 裡面所有的圖片檔名
    # 同時檢查 train 和 val 子資料夾
    chunk_images = glob(os.path.join(chunk_path, "**", "images", "*.jpg"), recursive=True)
    
    print(f"📦 準備從 {TARGET_CHUNK} 的源頭清理 {len(chunk_images)} 組檔案...")

    removed_count = 0
    
    for img_path in chunk_images:
        filename = os.path.basename(img_path)
        basename = os.path.splitext(filename)[0]
        
        # 定義可能的源頭路徑 (原始圖片、原始標籤、背景圖)
        possible_sources = [
            os.path.join(SOURCE_IMG_DIR, filename),      # 原始圖
            os.path.join(SOURCE_IMG_DIR, basename + ".txt"), # 原始標籤
            os.path.join(BACKGROUND_DIR, filename)       # 如果是背景圖
        ]
        
        found_and_moved = False
        for src in possible_sources:
            if os.path.exists(src):
                # 移動到回收站而非直接刪除
                dst = os.path.join(TRASH_DIR, os.path.basename(src))
                shutil.move(src, dst)
                found_and_moved = True
        
        if found_and_moved:
            removed_count += 1
            print(f"🗑️ 已移除: {filename}")

    print("\n" + "="*30)
    print(f"✅ 清理完成！")
    print(f"總計從源頭移除了 {removed_count} 組圖文資料。")
    print(f"檔案目前存放在: {TRASH_DIR} (確認沒問題後可手動清空)")
    print("="*30)

if __name__ == "__main__":
    cleanup_source_files()

📦 準備從 chunk_4 的源頭清理 50 組檔案...
🗑️ 已移除: 1042.jpg
🗑️ 已移除: 1044.jpg
🗑️ 已移除: 1061.jpg
🗑️ 已移除: 1068.jpg
🗑️ 已移除: 1118.jpg
🗑️ 已移除: 1126.jpg
🗑️ 已移除: 1130.jpg
🗑️ 已移除: 1135.jpg
🗑️ 已移除: 1138.jpg
🗑️ 已移除: 1149.jpg
🗑️ 已移除: 1155.jpg
🗑️ 已移除: 216.jpg
🗑️ 已移除: 280.jpg
🗑️ 已移除: 295.jpg
🗑️ 已移除: 408.jpg
🗑️ 已移除: 44.jpg
🗑️ 已移除: 565.jpg
🗑️ 已移除: 568.jpg
🗑️ 已移除: 583.jpg
🗑️ 已移除: 588.jpg
🗑️ 已移除: 590.jpg
🗑️ 已移除: 591.jpg
🗑️ 已移除: 593.jpg
🗑️ 已移除: 606.jpg
🗑️ 已移除: 634.jpg
🗑️ 已移除: 639.jpg
🗑️ 已移除: 661.jpg
🗑️ 已移除: 668.jpg
🗑️ 已移除: 772.jpg
🗑️ 已移除: 822.jpg
🗑️ 已移除: 857.jpg
🗑️ 已移除: 882.jpg
🗑️ 已移除: 888.jpg
🗑️ 已移除: 914.jpg
🗑️ 已移除: 929.jpg
🗑️ 已移除: 937.jpg
🗑️ 已移除: 942.jpg
🗑️ 已移除: 963.jpg
🗑️ 已移除: 989.jpg
🗑️ 已移除: 996.jpg
🗑️ 已移除: 1031.jpg
🗑️ 已移除: 655.jpg
🗑️ 已移除: 748.jpg
🗑️ 已移除: 886.jpg
🗑️ 已移除: 928.jpg
🗑️ 已移除: 945.jpg
🗑️ 已移除: 954.jpg
🗑️ 已移除: 970.jpg
🗑️ 已移除: 990.jpg
🗑️ 已移除: 992.jpg

✅ 清理完成！
總計從源頭移除了 50 組圖文資料。
檔案目前存放在: ./dataset_trash_bin (確認沒問題後可手動清空)
